In [15]:
import re

In [16]:
def read_data(path):
    with open(path) as f:
        return [x.strip() for x in f.readlines()]

In [17]:
def segment_example(example):
    segment = []
    segments = []

    example = example.split()

    for i, src_token in enumerate(example):
        if src_token in ['.', '?', '؟', '!']:
            if len(segment) == 0:
                if segments:
                    segments[-1].append(src_token)

                else: # it happened in the beginning
                    segment.append(src_token)

            else:
                segment.append(src_token)
                segments.append(segment)
                segment = []
        else:
            segment.append(src_token)
    
    if segment:
        if len(segment) < 2 and segments:
            segments[-1] += segment
        else:
            segments.append(segment)
    
    return segments

In [18]:
def verify(unsegmented_dataset, segmented_dataset):
    len(segmented_dataset) == len(unsegmented_dataset)

    RE_SPACE = re.compile(r' +')

    for i, unseg_src in enumerate(unsegmented_dataset):
        seg_example = segmented_dataset[i]

        seg_src = [word for seg in seg_example for word in seg]

        unseg_src = RE_SPACE.sub(' ', unseg_src)

        seg_src = RE_SPACE.sub(' ', ' '.join(seg_src))

        if unseg_src != seg_src:
            import pdb; pdb.set_trace()
        assert seg_src == unseg_src



In [19]:
def segment_data(dataset):
    segmented_data = {}

    for i, example in enumerate(dataset):
        segmented_example = segment_example(example)
        segmented_data[i] = segmented_example

    verify(unsegmented_dataset=dataset,
           segmented_dataset=segmented_data)

    return segmented_data

In [20]:
def get_stats(dataset):

    num_essays = len(dataset)
    flatten_dataset = [example for segments in dataset.values() for example in segments]
    num_segments = len(flatten_dataset)
    src_lengths = [len(ex) for ex in flatten_dataset]
    max_length = max(src_lengths)
    min_length = min(src_lengths)
    avg_length = sum(src_lengths) / len(src_lengths)

    print(f'Num Essays:   {num_essays}')
    print(f'Num Segments: {num_segments}')
    print(f'Min Length:   {min_length}')
    print(f'Max Length:   {max_length}')
    print(f'Avg Length:   {avg_length:.2f}')
    print()

In [21]:
def write_data(src_path, ids_path, split, data):
    RE_SPACE = re.compile(r' +')
    with open(src_path, mode='w') as f1, open(ids_path, mode='w') as f2:
        for ex_num in data:
            example = data[ex_num]
            for src_seg in example:
                src_seg = RE_SPACE.sub(' ', ' '.join(src_seg))
                f1.write(src_seg)
                f1.write('\n')
                f2.write(f'{split.upper()}-{ex_num}')
                f2.write('\n')


In [24]:

for dataset in ['zaebuc-w1', 'zaebuc-w2']:
    for split in ['train', 'dev', 'test']:
        print(f'{dataset} EN {split}')
        unseg_data = read_data(f'/scratch/ba63/zaebuc-lrec-2026/public-release/written/{dataset}/en/gec/{split}.raw.tok')[1:]
        seg_data = segment_data(unseg_data)
        get_stats(seg_data)
        write_data(src_path=f'/scratch/ba63/zaebuc-lrec-2026/expriments/written/gec/data/{dataset}/en/{split}.raw.tok.seg',
                   split=split,
                   ids_path=f'/scratch/ba63/zaebuc-lrec-2026/expriments/written/gec/data/{dataset}/en/{split}.raw.tok.seg.ids',
                   data=seg_data)


zaebuc-w1 EN train
Num Essays:   272
Num Segments: 2328
Min Length:   2
Max Length:   287
Avg Length:   28.49

zaebuc-w1 EN dev
Num Essays:   58
Num Segments: 526
Min Length:   3
Max Length:   288
Avg Length:   28.28

zaebuc-w1 EN test
Num Essays:   58
Num Segments: 510
Min Length:   2
Max Length:   300
Avg Length:   27.53

zaebuc-w2 EN train
Num Essays:   216
Num Segments: 1804
Min Length:   2
Max Length:   170
Avg Length:   26.67

zaebuc-w2 EN dev
Num Essays:   46
Num Segments: 382
Min Length:   5
Max Length:   218
Avg Length:   27.52

zaebuc-w2 EN test
Num Essays:   46
Num Segments: 415
Min Length:   2
Max Length:   138
Avg Length:   24.62

